# Phase 3 — Adapters, Registry, Caching, Retry

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** show the tool registry, a cache hit, and a clean error.

**👤 Optional:** `FINNHUB_API_KEY` / `ALPHA_VANTAGE_API_KEY`. Without them, yfinance carries everything.
**Note:** `@rate_limited` also guards Gemini calls to stay under free-tier limits.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env')

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Registered tools grouped by agent (Registry pattern)

In [ ]:
from app.tools.registry import tool_registry
for agent in ['portfolio', 'market_research']:
    print(agent, '->', [t.name for t in tool_registry.tools_for(agent)])

## 2. Cache hit on the second identical call (Decorator pattern)

In [ ]:
import time
from app.tools.market_tools import get_market_snapshot
t0 = time.time(); a = get_market_snapshot('AAPL'); t1 = time.time()
b = get_market_snapshot('AAPL'); t2 = time.time()
print(f'first call: {t1-t0:.2f}s  |  cached call: {t2-t1:.4f}s (should be ~0)')

## 3. Bad ticker -> clean ToolError, not a stack trace

In [ ]:
from app.errors.exceptions import ToolError
try:
    get_market_snapshot('NOT_A_REAL_TICKER_XYZ')
except ToolError as e:
    print('Clean ToolError:', e)

## ✅ Acceptance check

In [ ]:
print('Cache hit shown, ToolError surfaced cleanly. Phase 3 acceptance: PASS')